# Spatial Constraints and Connectivity Analysis

This notebook demonstrates spatial constraints, adjacency requirements, and connectivity analysis using `ws3`.

> **Prerequisites**: Completion of `070_ws3_quickstart_complete_workflow.ipynb`

## What You'll Learn

- How to add adjacency constraints to optimization
- How to enforce contiguous area requirements
- How to analyze spatial connectivity
- How to visualize spatial patterns

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
from ws3.common import rasterize_stands, hash_dt
from ws3.spatial import ForestRaster
from util import schedule_harvest_areacontrol, compile_scenario, plot_scenario

In [ ]:
# Model parameters
base_year = 2020
horizon = 5
period_length = 10
max_age = 1000

print(f"Model Parameters:")
print(f"  Base Year: {base_year}")
print(f"  Horizon: {horizon} periods")
print(f"  Period Length: {period_length} years")

In [ ]:
# Load stand inventory
stands_path = "data/shp/tsa24_clipped.shp/stands.shp"
stands = gpd.read_file(stands_path)

print(f"Loaded {len(stands)} stands")
print(f"Columns: {list(stands.columns)}")
stands.head()

In [ ]:
# Rasterize the stand inventory
RASTER_DIR = "data/raster_spatial_demo"
RASTER_DIR_PATH = Path(RASTER_DIR)
RASTER_DIR_PATH.mkdir(parents=True, exist_ok=True)
INVENTORY_TIF = RASTER_DIR_PATH / 'tsa24_inventory.tif'

# Create hash table for spatial indexing
hash_table = hash_dt(stands, include=['theme0', 'theme1', 'theme2', 'theme3', 'theme4', 'age'])

# Rasterize stands
raster = rasterize_stands(
    stands,
    hash_table,
    output_path=str(INVENTORY_TIF),
    resolution=100,
    crs=stands.crs
)

print(f"Rasterized inventory: {raster.shape}")

In [ ]:
# Create ForestRaster object
fr = ForestRaster(INVENTORY_TIF)

print(f"ForestRaster loaded:")
print(f"  Shape: {fr.shape}")
print(f"  Resolution: {fr.resolution}")
print(f"  CRS: {fr.crs}")

In [ ]:
# Visualize the inventory raster
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Theme 0 (landbase)
axes[0].imshow(fr.data[0], cmap='viridis')
axes[0].set_title('Theme 0: Landbase')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')

# Theme 1 (ownership)
axes[1].imshow(fr.data[1], cmap='tab10')
axes[1].set_title('Theme 1: Ownership')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')

# Theme 2 (forest type)
axes[2].imshow(fr.data[2], cmap='tab20')
axes[2].set_title('Theme 2: Forest Type')
axes[2].set_xlabel('X')
axes[2].set_ylabel('Y')

plt.tight_layout()
plt.show()

In [ ]:
# Define adjacency matrix
# Two pixels are adjacent if they share an edge (4-connectivity)
def get_adjacency_matrix(raster_shape):
    """Create adjacency matrix for 4-connected grid."""
    n = raster_shape[0] * raster_shape[1]
    adj_matrix = np.zeros((n, n))
    
    for i in range(raster_shape[0]):
        for j in range(raster_shape[1]):
            idx = i * raster_shape[1] + j
            
            # Right neighbor
            if j < raster_shape[1] - 1:
                right_idx = i * raster_shape[1] + (j + 1)
                adj_matrix[idx, right_idx] = 1
                adj_matrix[right_idx, idx] = 1
            
            # Down neighbor
            if i < raster_shape[0] - 1:
                down_idx = (i + 1) * raster_shape[1] + j
                adj_matrix[idx, down_idx] = 1
                adj_matrix[down_idx, idx] = 1
    
    return adj_matrix

adj_matrix = get_adjacency_matrix(fr.shape)
print(f"Adjacency matrix shape: {adj_matrix.shape}")
print(f"Number of adjacent pairs: {np.sum(adj_matrix) // 2}")

In [ ]:
# Define contiguous area constraint
def check_contiguity(binary_mask, min_area=10):
    """Check if all patches meet minimum area requirement."""
    from scipy import ndimage
    
    # Label connected components
    labeled, num_features = ndimage.label(binary_mask)
    
    # Count patch sizes
    patch_sizes = ndimage.sum(binary_mask, labeled, range(1, num_features + 1))
    
    # Find small patches
    small_patches = np.where(patch_sizes < min_area)[0] + 1
    
    if len(small_patches) > 0:
        print(f"Found {len(small_patches)} patches below minimum area ({min_area} cells)")
        for patch_id in small_patches[:5]:  # Show first 5
            patch_area = patch_sizes[patch_id - 1]
            print(f"  Patch {patch_id}: {patch_area} cells")
        return False
    else:
        print(f"All {num_features} patches meet minimum area requirement")
        return True

# Example: Check a hypothetical harvest mask
# (In practice, you'd check the actual harvest schedule)
print("Contiguity check function defined")

In [ ]:
# Analyze spatial connectivity
def calculate_connectivity_metrics(raster_data, threshold=0):
    """Calculate connectivity metrics for a binary raster."""
    from scipy import ndimage
    
    binary_mask = raster_data > threshold
    
    # Label connected components
    labeled, num_features = ndimage.label(binary_mask)
    
    # Calculate metrics
    metrics = {
        'total_cells': np.sum(binary_mask),
        'num_patches': num_features,
        'mean_patch_size': np.mean(ndimage.sum(binary_mask, labeled, range(1, num_features + 1))),
        'largest_patch': np.max(ndimage.sum(binary_mask, labeled, range(1, num_features + 1))) if num_features > 0 else 0,
        'connectivity_ratio': np.sum(binary_mask) / (raster_data.shape[0] * raster_data.shape[1])
    }
    
    return metrics

# Example metrics for theme 2 (forest type)
metrics = calculate_connectivity_metrics(fr.data[2], threshold=0)
print("Spatial Connectivity Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Visualize spatial patterns with adjacency
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Show adjacency relationships
axes[0].imshow(adj_matrix[:100, :100], cmap='binary', interpolation='nearest')
axes[0].set_title('Adjacency Matrix (First 100 Cells)')
axes[0].set_xlabel('Cell Index')
axes[0].set_ylabel('Cell Index')

# Show spatial distribution
axes[1].imshow(fr.data[2], cmap='tab20')
axes[1].set_title('Forest Type Distribution (Theme 2)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')

plt.tight_layout()
plt.show()

In [ ]:
# Summary of spatial analysis
print("=" * 60)
print("SPATIAL CONSTRAINTS SUMMARY")
print("=" * 60)
print(f"Raster Dimensions: {fr.shape}")
print(f"Resolution: {fr.resolution} m")
print(f"Total Cells: {fr.shape[0] * fr.shape[1]}")
print(f"Adjacent Pairs: {np.sum(adj_matrix) // 2}")
print("=" * 60)
print("\nSpatial constraints enable:")
print("  - Adjacency requirements between harvest blocks")
print("  - Minimum contiguous area for harvesting")
print("  - Spatial connectivity analysis")
print("  - Buffer zone enforcement")